In [1]:
!pip install -q unidecode tqdm

import os
import json
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.parametrizations import weight_norm
from torch.cuda.amp import GradScaler, autocast
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
random.seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 10.3 MB/s eta 0:00:00


In [2]:
def split_data_by_source(file_path, train_ratio=0.8, val_ratio=0.1):
    data_by_source = defaultdict(list)
    print(f"Đang đọc dữ liệu từ: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line)
                text = item.get('segmented_text', '').strip()
                source = item.get('source', 'unknown')
                if text: 
                    data_by_source[source].append(text)
            except json.JSONDecodeError:
                continue

    train_texts, val_texts, test_texts = [], [], []
    
    for source, texts in data_by_source.items():
        random.shuffle(texts)
        n = len(texts)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        
        train_texts.extend(texts[:n_train])
        val_texts.extend(texts[n_train:n_train + n_val])
        test_texts.extend(texts[n_train + n_val:])
        
        print(f" - Source '{source}': {n} mẫu (Train: {n_train}, Val: {n_val}, Test: {n - n_train - n_val})")
        
    print("-" * 50)
    print(f"Tổng cộng -> Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
    return train_texts, val_texts, test_texts

# Đường dẫn file Kaggle của bạn
dataset_path = '/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl'
train_texts, val_texts, test_texts = split_data_by_source(dataset_path)

Đang đọc dữ liệu từ: /kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl
 - Source 'news_dantri': 186831 mẫu (Train: 149464, Val: 18683, Test: 18684)
 - Source 'news_thanhnien': 192765 mẫu (Train: 154212, Val: 19276, Test: 19277)
 - Source 'news_vnexpress': 123522 mẫu (Train: 98817, Val: 12352, Test: 12353)
 - Source 'forum_voz': 294343 mẫu (Train: 235474, Val: 29434, Test: 29435)
--------------------------------------------------
Tổng cộng -> Train: 637967 | Val: 79745 | Test: 79749


In [3]:
class Vocabulary:
    def __init__(self, max_size=None, min_freq=2):
        self.pad_token, self.pad_idx = '<pad>', 0
        self.unk_token, self.unk_idx = '<unk>', 1
        self.sos_token, self.sos_idx = '<sos>', 2
        self.eos_token, self.eos_idx = '<eos>', 3
        
        self.word2idx = {self.pad_token: 0, self.unk_token: 1, self.sos_token: 2, self.eos_token: 3}
        self.idx2word = {0: self.pad_token, 1: self.unk_token, 2: self.sos_token, 3: self.eos_token}
        self.max_size = max_size
        self.min_freq = min_freq
        self.word_freqs = Counter()

    def build_vocab(self, texts):
        print("Đang xây dựng từ điển từ tập Train...")
        for text in texts:
            self.word_freqs.update(text.split())
        
        valid_words = [w for w, freq in self.word_freqs.items() if freq >= self.min_freq]
        valid_words = sorted(valid_words, key=lambda w: self.word_freqs[w], reverse=True)
        
        if self.max_size:
            valid_words = valid_words[:self.max_size - 4]
            
        for word in valid_words:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word
        print(f"Hoàn tất! Kích thước từ điển: {len(self.word2idx)} từ.")

    def encode(self, text, add_sos_eos=True):
        tokens = text.split()
        seq = [self.word2idx.get(w, self.unk_idx) for w in tokens]
        if add_sos_eos:
            seq = [self.sos_idx] + seq + [self.eos_idx]
        return seq

    def decode(self, indices):
        return " ".join([self.idx2word.get(idx, self.unk_token) for idx in indices])

class TCNDataset(Dataset):
    def __init__(self, texts, vocab, seq_length=64, name="Dataset"):
        self.vocab = vocab
        self.seq_length = seq_length
        self.data = []
        
        print(f"Đang xử lý Sliding Window cho tập {name}...")
        for text in texts:
            encoded = self.vocab.encode(text, add_sos_eos=True)
            for i in range(0, len(encoded) - seq_length):
                chunk = encoded[i : i + seq_length + 1]
                if len(chunk) == seq_length + 1:
                    self.data.append(chunk)
        print(f"Tập {name} tạo được {len(self.data)} sequences.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chunk = self.data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

In [4]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.act1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.act1, self.drop1, self.conv2, self.chomp2, self.act2, self.drop2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.GELU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            padding = (kernel_size - 1) * dilation_size
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size, padding=padding, dropout=dropout))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

class TCNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, num_channels, kernel_size=2, dropout=0.2):
        super(TCNLanguageModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.tcn = TemporalConvNet(embed_size, num_channels, kernel_size=kernel_size, dropout=dropout)
        
        self.decoder = nn.Linear(num_channels[-1], vocab_size)
        if num_channels[-1] == embed_size:
            self.decoder.weight = self.embedding.weight 

    @autocast()
    # SỬA Ở ĐÂY: Nhận thêm targets
    def forward(self, x, targets=None): 
        emb = self.embedding(x)
        y = emb.transpose(1, 2) 
        y = self.tcn(y) 
        y = y.transpose(1, 2)
        logits = self.decoder(y)
        
        # SỬA Ở ĐÂY: Tính Loss ngay trên từng GPU nếu có targets
        if targets is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=0)
            # Ép phẳng logits và targets để tính loss
            loss = loss_fct(logits.view(-1, logits.size(-1)), targets.view(-1))
            return loss, logits
            
        return logits

In [5]:
def evaluate_model(model, dataloader, criterion, device='cuda'):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
            total_loss += loss.item()
    return total_loss / len(dataloader)

def train_tcn_model_with_val(model, train_loader, val_loader, epochs=5, lr=1e-3, device='cuda'):
    if torch.cuda.device_count() > 1:
        print(f"Đang sử dụng {torch.cuda.device_count()} GPUs với DataParallel!")
        model = nn.DataParallel(model)
        
    model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=0) 
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    
    total_steps = len(train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1)
    scaler = GradScaler()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            # SỬA Ở ĐÂY: Truyền cả targets vào model
            loss, logits = model(inputs, targets)
            
            # Nếu dùng DataParallel, loss trả về sẽ là 1 vector gồm [loss_gpu0, loss_gpu1]
            # Ta cần lấy trung bình cộng để ra loss duy nhất cho batch này
            loss = loss.mean()
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
            
        train_loss = total_loss / len(train_loader)
        train_ppl = math.exp(train_loss) if train_loss < 20 else float('inf')
        
        # Gọi hàm đánh giá trên tập Validation
        val_loss = evaluate_model(model, val_loader, criterion, device)
        val_ppl = math.exp(val_loss) if val_loss < 20 else float('inf')
        
        print(f"=> Kết thúc Epoch {epoch+1} | Train Loss: {train_loss:.4f} (PPL: {train_ppl:.2f}) | Val Loss: {val_loss:.4f} (PPL: {val_ppl:.2f})\n")

    return model

In [ ]:
# 1. Khởi tạo và xây dựng từ điển CHỈ TỪ TẬP TRAIN
vocab = Vocabulary(max_size=30000, min_freq=3)
vocab.build_vocab(train_texts)

# 2. Khởi tạo Dataset
seq_length = 32
train_dataset = TCNDataset(train_texts, vocab, seq_length=seq_length, name="Train")
val_dataset = TCNDataset(val_texts, vocab, seq_length=seq_length, name="Validation")
# Có thể khởi tạo test_dataset nếu bạn muốn đánh giá tập test sau này
test_dataset = TCNDataset(test_texts, vocab, seq_length=seq_length, name="Test")

# 3. DataLoaders (Tối ưu cho Dual T4 Kaggle)
batch_size = 1024
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# 4. Định nghĩa siêu tham số (Đã sửa lỗi chiều dữ liệu ở đây)
vocab_size = len(vocab.word2idx)
embed_size = 256
# ĐẢM BẢO TẦNG CUỐI (256) BẰNG VỚI EMBED_SIZE (256) ĐỂ DÙNG WEIGHT TYING
num_channels = [128, 256, 512, 256] 
kernel_size = 3

model = TCNLanguageModel(vocab_size=vocab_size, embed_size=embed_size, 
                         num_channels=num_channels, kernel_size=kernel_size)

# 5. Huấn luyện!
print("Bắt đầu huấn luyện mô hình trên Dual T4 GPUs...")
trained_model = train_tcn_model_with_val(model, train_loader, val_loader, epochs=3, lr=2e-3)



In [ ]:
def beam_search_decode(model, vocab, start_text, beam_width=3, max_len=20, length_penalty_alpha=0.7, device='cuda'):
    model.eval()
    
    start_text = start_text.lower() 
    
    raw_encoded = vocab.encode(start_text, add_sos_eos=False)
    if vocab.unk_idx in raw_encoded:
        print(f" Cảnh báo: Có từ trong chuỗi '{start_text}' không nằm trong từ điển và bị biến thành <unk>!")
    
    input_ids = [vocab.sos_idx] + raw_encoded
    beams = [(input_ids, 0.0)]
    
    for step in range(max_len):
        new_beams = []
        for seq, score in beams:
            if seq[-1] == vocab.eos_idx:
                new_beams.append((seq, score))
                continue
                
            input_tensor = torch.tensor([seq], dtype=torch.long).to(device)
            
            with torch.no_grad():
                with torch.autocast(device_type=device): 
                    logits = model(input_tensor)
                    last_token_logits = logits[0, -1, :]
                    
                    for token_idx in set(seq):
                        last_token_logits[token_idx] -= 2.0 
                    
                    log_probs = F.log_softmax(last_token_logits, dim=-1)
            
            top_log_probs, top_indices = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                next_token = top_indices[i].item()
                next_log_prob = top_log_probs[i].item()
                
                new_seq = seq + [next_token]
                length_penalty = ((5 + len(new_seq)) / 6) ** length_penalty_alpha
                new_score = score + (next_log_prob / length_penalty)
                new_beams.append((new_seq, new_score))
                
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        # Dừng nếu tất cả các beam tốt nhất đều kết thúc
        if all(b[0][-1] == vocab.eos_idx for b in beams):
            break
            
    best_seq = beams[0][0]
    filtered_seq = [idx for idx in best_seq if idx not in [vocab.sos_idx, vocab.eos_idx, vocab.pad_idx]]
    return vocab.decode(filtered_seq).replace("_", " ")

prompt = "trường đại học"
print(f"\nPrompt: {prompt.replace('_', ' ')}")
predicted_text = beam_search_decode(trained_model, vocab, prompt, beam_width=3, max_len=20)
print(f"Dự đoán Next Word: {predicted_text}")